# Titanic Survival Prediction — Data Preprocessing

This notebook cleans `train.csv` into a single standardized Pandas DataFrame that all predictive models (ANN, kNN, XGBoost) will share.

Cleaning strategy:
- Drop the non-predictive identifier and noise / high-missing columns entirely: `PassengerId`, `Name`, `Ticket`, and `Cabin`.
- Drop rows with null `Age` values.
- Drop the small number of rows with missing `Embarked` so the final DataFrame has no missing values.
- Encode the remaining categorical columns (`Sex`, `Embarked`) so the DataFrame is fully numeric and model-ready.
- Keep the remaining columns with predictive potential intact (e.g. `Sex`, `Pclass`, `Age`).

In [48]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/titanic/train.csv')
df.shape

(891, 12)

## 1. Initial Inspection

Examine data types, missing values, and completeness before making any cleaning decisions.

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [50]:
# Missing value counts and percentages per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_summary

,missing_count,missing_pct
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [51]:
# Row-level completeness (share of rows with zero missing values across all columns)
complete_rows = (df.notnull().all(axis=1)).sum()
complete_rows_pct = round(complete_rows / len(df) * 100, 2)
avg_completeness = round(df.notnull().mean().mean() * 100, 2)
print(f"Total records: {len(df)}")
print(f"Fully complete records: {complete_rows} ({complete_rows_pct}%)")
print(f"Average column completeness: {avg_completeness}%")

Total records: 891
Fully complete records: 183 (20.54%)
Average column completeness: 91.9%


`Cabin` is missing for the vast majority of passengers, `Age` is missing for a meaningful minority, and `Embarked` is missing for only two records. `PassengerId` is a row identifier with no predictive meaning, and `Name` and `Ticket` are near-unique identifiers that add noise rather than predictive value, while `Cabin`'s heavy missingness makes it unusable without aggressive imputation, so all four are dropped outright.

## 2. Drop Identifier and Noise / High-Missing Columns (`PassengerId`, `Name`, `Ticket`, `Cabin`)

`PassengerId` is only a row identifier, `Name` and `Ticket` are effectively unique per passenger and only introduce noise, and `Cabin` is missing for the large majority of records. None of these offers reliable predictive signal, so they are removed entirely rather than imputed. This keeps the variables with predictive potential (e.g. `Sex`, `Pclass`, `Age`).

In [52]:
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
for col in drop_cols:
    missing_pct = round(df[col].isnull().mean() * 100, 2)
    print(f"{col}: {df[col].isnull().sum()} missing of {len(df)} records ({missing_pct}%)")


df_clean = df.drop(columns=drop_cols)
df_clean.shape

PassengerId: 0 missing of 891 records (0.0%)
Name: 0 missing of 891 records (0.0%)
Ticket: 0 missing of 891 records (0.0%)
Cabin: 687 missing of 891 records (77.1%)


(891, 8)

## 3. Drop Rows with Missing `Age`

Rather than imputing `Age`, rows with a missing value are dropped. `Age` is an important predictor of survival, and dropping these rows avoids introducing synthetic values into the distribution.

In [53]:
rows_before = len(df_clean)
age_missing = df_clean['Age'].isnull().sum()
age_missing_pct = round(age_missing / rows_before * 100, 2)
print(f"Rows with missing Age: {age_missing} of {rows_before} ({age_missing_pct}%)")

df_clean = df_clean.dropna(subset=['Age']).reset_index(drop=True)
rows_after = len(df_clean)
print(f"Rows before: {rows_before}, rows after: {rows_after}, rows dropped: {rows_before - rows_after}")

Rows with missing Age: 177 of 891 (19.87%)
Rows before: 891, rows after: 714, rows dropped: 177


## 4. Remove Rows with Missing `Embarked`

The goal is a DataFrame with no missing values that every model can share. Only two rows are missing `Embarked`, so these rows are dropped as well (consistent with how `Age` is handled), leaving zero missing values without inventing any data.

In [54]:
embarked_missing = df_clean['Embarked'].isnull().sum()
print(f"Rows with missing Embarked: {embarked_missing}")

df_clean = df_clean.dropna(subset=['Embarked']).reset_index(drop=True)
print(f"Rows after dropping missing Embarked: {len(df_clean)}")
df_clean.isnull().sum()

Rows with missing Embarked: 2
Rows after dropping missing Embarked: 712


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## 5. Verify Final Cleaned DataFrame

The resulting `df_clean` drops the noise / high-missing columns (`Name`, `Ticket`, `Cabin`), contains no missing values in any remaining column, and is the single shared DataFrame to be used for training the ANN, kNN, and XGBoost models.

In [55]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Sex       712 non-null    str    
 3   Age       712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  712 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 48.8 KB


In [56]:
df_clean.describe(include='all')

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
count,712.000000,712.000000,712,712.000000,712.000000,712.000000,712.000000,712
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S
freq,NaN,NaN,453,NaN,NaN,NaN,NaN,554
mean,0.404494,2.240169,NaN,29.642093,0.514045,0.432584,34.567251,NaN
std,0.491139,0.836854,NaN,14.492933,0.930692,0.854181,52.938648,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN
25%,0.000000,1.000000,NaN,20.000000,0.000000,0.000000,8.050000,NaN
50%,0.000000,2.000000,NaN,28.000000,0.000000,0.000000,15.645850,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,1.000000,33.000000,NaN


In [57]:
df_clean.head(10)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
5,0,1,male,54.0,0,0,51.8625,S
6,0,3,male,2.0,3,1,21.0750,S
7,1,3,female,27.0,0,2,11.1333,S
8,1,2,female,14.0,1,0,30.0708,C
9,1,3,female,4.0,1,1,16.7000,S


## 6. Encode Categorical Variables

kNN, XGBoost, and the neural network all require numeric inputs, so the two remaining categorical columns are encoded. `Sex` is binary and mapped to 0/1, while `Embarked` has three unordered categories and is one-hot encoded so that no false ordinal relationship is implied between ports. After this step every column in `df_clean` is numeric and ready for model training.

In [58]:
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], prefix='Embarked', dtype=int)

print("Encoded columns:", list(df_clean.columns))
print(df_clean.dtypes)
df_clean.head()

Encoded columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_C      int64
Embarked_Q      int64
Embarked_S      int64
dtype: object


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,0,1
1,1,1,1,38.0,1,0,71.2833,1,0,0
2,1,3,1,26.0,0,0,7.9250,0,0,1
3,1,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,0,1


## 7. Summary Statistics for the Paper

In [59]:
final_complete_pct = round((df_clean.notnull().all(axis=1)).sum() / len(df_clean) * 100, 2)
final_avg_completeness = round(df_clean.notnull().mean().mean() * 100, 2)
survival_rate = round(df_clean['Survived'].mean() * 100, 2)
sex_dist = (df_clean['Sex'].map({0: 'male', 1: 'female'}).value_counts(normalize=True) * 100).round(2)
pclass_dist = (df_clean['Pclass'].value_counts(normalize=True).sort_index() * 100).round(2)

print(f"Original records: {len(df)}")
print(f"Cleaned records: {len(df_clean)} ({round(len(df_clean)/len(df)*100, 2)}% retained)")
print(f"Rows dropped due to missing Age: {rows_before - rows_after} ({age_missing_pct}%)")
print(f"Fully complete records after cleaning: {final_complete_pct}%")
print(f"Average completeness after cleaning: {final_avg_completeness}%")
print(f"Overall survival rate: {survival_rate}%")
print(f"Sex distribution: {sex_dist.to_dict()}")
print(f"Pclass distribution: {pclass_dist.to_dict()}")

Original records: 891
Cleaned records: 712 (79.91% retained)
Rows dropped due to missing Age: 177 (19.87%)
Fully complete records after cleaning: 100.0%
Average completeness after cleaning: 100.0%
Overall survival rate: 40.45%
Sex distribution: {'male': 63.62, 'female': 36.38}
Pclass distribution: {1: 25.84, 2: 24.3, 3: 49.86}


In [60]:
df_clean.head


<bound method NDFrame.head of      Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_C  \
0           0       3    0  22.0      1      0   7.2500           0   
1           1       1    1  38.0      1      0  71.2833           1   
2           1       3    1  26.0      0      0   7.9250           0   
3           1       1    1  35.0      1      0  53.1000           0   
4           0       3    0  35.0      0      0   8.0500           0   
..        ...     ...  ...   ...    ...    ...      ...         ...   
707         0       3    1  39.0      0      5  29.1250           0   
708         0       2    0  27.0      0      0  13.0000           0   
709         1       1    1  19.0      0      0  30.0000           0   
710         1       1    0  26.0      0      0  30.0000           1   
711         0       3    0  32.0      0      0   7.7500           0   

     Embarked_Q  Embarked_S  
0             0           1  
1             0           0  
2             0           1

`df_clean` is the standardized, cleaned DataFrame ready to be shared across the ANN, kNN, and XGBoost training pipelines.

## 8. Output Cleaned Training Data

In [61]:
# Output the cleaned training datas as a csv
df_clean.to_csv('../data/titanic/cleaned_train.csv', index = False)